<a href="https://colab.research.google.com/github/ridahafeez786/AI_Bootcamp_Lab-work/blob/main/CoT_ToT_GoT_Lab_Solution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install -U "transformers>=4.37.0" accelerate sentencepiece pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 67.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.5 which is incompatible.


In [2]:
from __future__ import annotations

import copy
import json
import re
from collections import Counter
from datetime import datetime
from typing import Any

import pandas as pd
import torch
from IPython.display import display
from transformers import AutoModelForCausalLM, AutoTokenizer

pd.set_option("display.max_colwidth", 120)

## 1. Load a small local Hugging Face model

In [3]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto",
)

print("Loaded:", MODEL_NAME)
print("Device:", model.device)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded: Qwen/Qwen2.5-0.5B-Instruct
Device: cpu


In [4]:
def ask_llm(
    prompt: str,
    *,
    do_sample: bool = False,
    temperature: float = 0.8,
    max_new_tokens: int = 700,
) -> str:
    messages = [
        {"role": "system", "content": "You are a careful scheduling and constraint-solving assistant."},
        {"role": "user", "content": prompt},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    kwargs = {
        "max_new_tokens": max_new_tokens,
        "do_sample": do_sample,
        "pad_token_id": tokenizer.eos_token_id,
    }
    if do_sample:
        kwargs.update({"temperature": temperature, "top_p": 0.9})

    with torch.no_grad():
        output = model.generate(**inputs, **kwargs)

    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

## 2. Problem statement, output schema and valid fallback

In [5]:
PROBLEM = """
Create a valid schedule for a one-day university AI Workshop.

Constraints:
- Budget must not exceed $5,000.
- All activities must remain between 09:00 and 17:00.
- Maximum participants: 120.
- Required exact session names:
  Opening Session
  Technical Session 1
  Technical Session 2
  Lunch Break
  Hands-on Lab
  Closing Session
- Available speakers: Speaker 1, Speaker 2 and Speaker 3 only.
- Each speaker may conduct at most two speaker-led sessions.
- Lunch must last exactly 60 minutes.
- Hands-on Lab must start at or after 14:00 and end at or before 16:00.
- Sessions must not overlap.
- A non-speaker setup or networking activity is allowed.
""".strip()

SCHEMA = """
Return only one JSON object:

{
  "method": "METHOD",
  "participants": 120,
  "schedule": [
    {"session": "Opening Session", "start": "09:00", "end": "09:30", "speaker": "Speaker 1"}
  ],
  "budget": [
    {"item": "Venue and utilities", "cost": 1000}
  ],
  "reasoning_summary": "Short explanation"
}

Use 24-hour HH:MM time. Use an empty speaker string for lunch or non-speaker activities.
Do not wrap the JSON in Markdown.
""".strip()

REFERENCE = {
    "method": "Reference",
    "participants": 120,
    "schedule": [
        {"session": "Opening Session", "start": "09:00", "end": "09:30", "speaker": "Speaker 1"},
        {"session": "Technical Session 1", "start": "09:30", "end": "10:45", "speaker": "Speaker 1"},
        {"session": "Technical Session 2", "start": "10:45", "end": "12:00", "speaker": "Speaker 2"},
        {"session": "Lunch Break", "start": "12:00", "end": "13:00", "speaker": ""},
        {"session": "Networking and Lab Setup", "start": "13:00", "end": "14:00", "speaker": ""},
        {"session": "Hands-on Lab", "start": "14:00", "end": "16:00", "speaker": "Speaker 3"},
        {"session": "Closing Session", "start": "16:00", "end": "17:00", "speaker": "Speaker 2"},
    ],
    "budget": [
        {"item": "Venue and utilities", "cost": 1000},
        {"item": "Lunch and refreshments", "cost": 1800},
        {"item": "Speaker honoraria", "cost": 1500},
        {"item": "Lab equipment, internet and support", "cost": 450},
        {"item": "Materials and certificates", "cost": 150},
        {"item": "Contingency", "cost": 100},
    ],
    "reasoning_summary": (
        "Morning sessions precede a one-hour lunch and setup period. "
        "The lab uses the full 14:00–16:00 allowed window, and the closing session ends at 17:00. "
        "Speaker workloads are 2, 2 and 1 sessions. Total budget is $5,000."
    ),
}

## 3. JSON parser and deterministic validator

In [6]:
def extract_json(text: str) -> dict[str, Any]:
    cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip(), flags=re.I)
    left, right = cleaned.find("{"), cleaned.rfind("}")
    if left == -1 or right == -1:
        raise ValueError("No JSON object found.")
    return json.loads(cleaned[left:right + 1])


def minutes(value: str) -> int:
    t = datetime.strptime(value, "%H:%M")
    return t.hour * 60 + t.minute


def validate(solution: dict[str, Any]) -> pd.DataFrame:
    rows = []

    def check(name, passed, detail):
        rows.append({"Constraint": name, "Passed": bool(passed), "Details": detail})

    participants = solution.get("participants")
    schedule = solution.get("schedule", [])
    budget = solution.get("budget", [])

    check(
        "Maximum 120 participants",
        isinstance(participants, (int, float)) and 0 < participants <= 120,
        f"participants={participants}",
    )

    parsed, errors = [], []
    for item in schedule:
        try:
            parsed.append({**item, "_start": minutes(item["start"]), "_end": minutes(item["end"])})
        except Exception as exc:
            errors.append(str(exc))

    check("Valid HH:MM times", not errors, "valid" if not errors else str(errors))

    required = {
        "Opening Session", "Technical Session 1", "Technical Session 2",
        "Lunch Break", "Hands-on Lab", "Closing Session",
    }
    present = {x.get("session", "") for x in schedule}
    missing = sorted(required - present)
    check("All required sessions", not missing, "none missing" if not missing else str(missing))

    within_day = bool(parsed) and all(540 <= x["_start"] < x["_end"] <= 1020 for x in parsed)
    check("Within 09:00–17:00", within_day, "all schedule entries checked")

    ordered = sorted(parsed, key=lambda x: x["_start"])
    overlaps = [
        f"{a.get('session')} / {b.get('session')}"
        for a, b in zip(ordered, ordered[1:])
        if b["_start"] < a["_end"]
    ]
    check("No overlapping sessions", not overlaps, "none" if not overlaps else str(overlaps))

    lunch = [x for x in parsed if x.get("session") == "Lunch Break"]
    lunch_ok = len(lunch) == 1 and lunch[0]["_end"] - lunch[0]["_start"] == 60
    check("Lunch exactly 60 minutes", lunch_ok, f"entries={len(lunch)}")

    lab = [x for x in parsed if x.get("session") == "Hands-on Lab"]
    lab_ok = len(lab) == 1 and lab[0]["_start"] >= 840 and lab[0]["_end"] <= 960
    check("Lab within 14:00–16:00", lab_ok, f"entries={len(lab)}")

    non_speaker = {"Lunch Break", "Networking and Lab Setup"}
    led = [x for x in schedule if x.get("session") not in non_speaker]
    assigned = [x.get("speaker", "") for x in led]
    allowed = {"Speaker 1", "Speaker 2", "Speaker 3"}

    check(
        "Only three available speakers",
        bool(assigned) and all(s in allowed for s in assigned) and len(set(assigned)) <= 3,
        f"used={sorted(set(assigned))}",
    )

    counts = Counter(assigned)
    overloaded = {s: n for s, n in counts.items() if n > 2}
    check("Maximum two sessions per speaker", not overloaded, f"counts={dict(counts)}")

    total, budget_errors = 0.0, []
    for item in budget:
        try:
            cost = float(item["cost"])
            if cost < 0:
                raise ValueError("negative cost")
            total += cost
        except Exception as exc:
            budget_errors.append(str(exc))

    check("Valid budget entries", not budget_errors, "valid" if not budget_errors else str(budget_errors))
    check("Budget at most $5,000", not budget_errors and total <= 5000, f"total=${total:,.2f}")

    return pd.DataFrame(rows)


def valid(solution: dict[str, Any]) -> bool:
    return bool(validate(solution)["Passed"].all())


def score(solution: dict[str, Any]) -> int:
    return int(validate(solution)["Passed"].sum())


def safe_candidate(raw: str, method: str) -> tuple[dict[str, Any], str]:
    try:
        candidate = extract_json(raw)
        candidate["method"] = method
        if valid(candidate):
            return candidate, "Valid model-generated solution"
    except Exception as exc:
        parse_error = str(exc)
    else:
        parse_error = "The model proposal violated one or more constraints."

    fallback = copy.deepcopy(REFERENCE)
    fallback["method"] = method
    fallback["reasoning_summary"] = (
        f"{method} was demonstrated, but its final generated JSON was not fully reliable. "
        f"The validated fallback was used. Reason: {parse_error}"
    )
    return fallback, "Validated fallback used"


def show_solution(solution: dict[str, Any], title: str):
    print("\\n" + title)
    print("=" * len(title))
    print("Participants:", solution.get("participants"))
    print("Reasoning summary:", solution.get("reasoning_summary", ""))
    print("\\nSchedule")
    display(pd.DataFrame(solution.get("schedule", [])))
    print("\\nBudget")
    budget_df = pd.DataFrame(solution.get("budget", []))
    display(budget_df)
    if not budget_df.empty:
        print(f"Total: ${budget_df['cost'].sum():,.2f}")
    print("\\nValidation")
    report = validate(solution)
    display(report)
    print("VALID:", bool(report["Passed"].all()))


assert valid(REFERENCE)
display(validate(REFERENCE))

,Constraint,Passed,Details
0,Maximum 120 participants,True,participants=120
1,Valid HH:MM times,True,valid
2,All required sessions,True,none missing
3,Within 09:00–17:00,True,all schedule entries checked
4,No overlapping sessions,True,none
5,Lunch exactly 60 minutes,True,entries=1
6,Lab within 14:00–16:00,True,entries=1
7,Only three available speakers,True,"used=['Speaker 1', 'Speaker 2', 'Speaker 3']"
8,Maximum two sessions per speaker,True,"counts={'Speaker 1': 2, 'Speaker 2': 2, 'Speaker 3': 1}"
9,Valid budget entries,True,valid


# Method 1 — Chain of Thought (CoT)

One linear planning path is used:

```text
fixed constraints → schedule → speakers → budget → final candidate
```

In [7]:
cot_prompt = f"""
Use one linear, step-by-step planning approach.

Order:
1. Place lunch and the lab.
2. Arrange the other required sessions.
3. Assign speakers.
4. Allocate the budget.
5. Check all constraints.
6. Return only the final JSON with a short reasoning_summary.

{PROBLEM}

{SCHEMA.replace("METHOD", "CoT")}
"""

cot_raw = ask_llm(cot_prompt, do_sample=False, max_new_tokens=850)
print(cot_raw)

cot_solution, cot_status = safe_candidate(cot_raw, "CoT")
print("\\nStatus:", cot_status)
show_solution(cot_solution, "Chain of Thought Result")

```json
{
  "method": "CoT",
  "participants": 120,
  "schedule": [
    {
      "session": "Opening Session",
      "start": "09:00",
      "end": "09:30",
      "speaker": "Speaker 1"
    },
    {
      "session": "Technical Session 1",
      "start": "09:30",
      "end": "10:00",
      "speaker": "Speaker 2"
    },
    {
      "session": "Technical Session 2",
      "start": "10:00",
      "end": "10:30",
      "speaker": "Speaker 2"
    },
    {
      "session": "Lunch Break",
      "start": "10:30",
      "end": "11:00",
      "speaker": ""
    },
    {
      "session": "Hands-on Lab",
      "start": "11:00",
      "end": "11:30",
      "speaker": ""
    },
    {
      "session": "Closing Session",
      "start": "11:30",
      "end": "12:00",
      "speaker": ""
    }
  ],
  "budget": [
    {
      "item": "Venue and utilities",
      "cost": 1000
    }
  ],
  "reasoning_summary": "The schedule adheres to the given constraints by ensuring that each session does not overlap, has a

,session,start,end,speaker
0,Opening Session,09:00,09:30,Speaker 1
1,Technical Session 1,09:30,10:45,Speaker 1
2,Technical Session 2,10:45,12:00,Speaker 2
3,Lunch Break,12:00,13:00,
4,Networking and Lab Setup,13:00,14:00,
5,Hands-on Lab,14:00,16:00,Speaker 3
6,Closing Session,16:00,17:00,Speaker 2


\nBudget


,item,cost
0,Venue and utilities,1000
1,Lunch and refreshments,1800
2,Speaker honoraria,1500
3,"Lab equipment, internet and support",450
4,Materials and certificates,150
5,Contingency,100


Total: $5,000.00
\nValidation


,Constraint,Passed,Details
0,Maximum 120 participants,True,participants=120
1,Valid HH:MM times,True,valid
2,All required sessions,True,none missing
3,Within 09:00–17:00,True,all schedule entries checked
4,No overlapping sessions,True,none
5,Lunch exactly 60 minutes,True,entries=1
6,Lab within 14:00–16:00,True,entries=1
7,Only three available speakers,True,"used=['Speaker 1', 'Speaker 2', 'Speaker 3']"
8,Maximum two sessions per speaker,True,"counts={'Speaker 1': 2, 'Speaker 2': 2, 'Speaker 3': 1}"
9,Valid budget entries,True,valid


VALID: True


# Method 2 — Tree of Thought (ToT)

Three independent candidate branches are generated. Python validates every branch and selects the strongest one.

In [8]:
strategies = {
    "Balanced": "Create a balanced day with clear transitions.",
    "Lab-first": "Reserve 14:00–16:00 for the lab first, then fit everything else.",
    "Speaker-efficient": "Prioritize a simple speaker allocation with no overload.",
}

tot_branches = []

for name, instruction in strategies.items():
    prompt = f"""
Generate one Tree of Thought candidate branch.

Strategy: {name}
Instruction: {instruction}

{PROBLEM}

Check the branch, then return only JSON.
{SCHEMA.replace("METHOD", "ToT")}
"""
    raw = ask_llm(prompt, do_sample=True, temperature=0.85, max_new_tokens=850)
    candidate, status = safe_candidate(raw, f"ToT — {name}")
    tot_branches.append({
        "name": name,
        "solution": candidate,
        "score": score(candidate),
        "valid": valid(candidate),
        "status": status,
    })

branch_table = pd.DataFrame([
    {
        "Branch": x["name"],
        "Validation Score": x["score"],
        "Valid": x["valid"],
        "Status": x["status"],
    }
    for x in tot_branches
])
display(branch_table)

best_branch = max(tot_branches, key=lambda x: (x["valid"], x["score"]))
tot_solution = copy.deepcopy(best_branch["solution"])
tot_solution["method"] = "ToT"
tot_solution["reasoning_summary"] = (
    f"Three branches were explored. The '{best_branch['name']}' branch was selected "
    f"with {best_branch['score']} passed validation checks. "
    + tot_solution.get("reasoning_summary", "")
)

show_solution(tot_solution, "Tree of Thought Result")

,Branch,Validation Score,Valid,Status
0,Balanced,11,True,Validated fallback used
1,Lab-first,11,True,Validated fallback used
2,Speaker-efficient,11,True,Validated fallback used


\nTree of Thought Result
Participants: 120
Reasoning summary: Three branches were explored. The 'Balanced' branch was selected with 11 passed validation checks. ToT — Balanced was demonstrated, but its final generated JSON was not fully reliable. The validated fallback was used. Reason: The model proposal violated one or more constraints.
\nSchedule


,session,start,end,speaker
0,Opening Session,09:00,09:30,Speaker 1
1,Technical Session 1,09:30,10:45,Speaker 1
2,Technical Session 2,10:45,12:00,Speaker 2
3,Lunch Break,12:00,13:00,
4,Networking and Lab Setup,13:00,14:00,
5,Hands-on Lab,14:00,16:00,Speaker 3
6,Closing Session,16:00,17:00,Speaker 2


\nBudget


,item,cost
0,Venue and utilities,1000
1,Lunch and refreshments,1800
2,Speaker honoraria,1500
3,"Lab equipment, internet and support",450
4,Materials and certificates,150
5,Contingency,100


Total: $5,000.00
\nValidation


,Constraint,Passed,Details
0,Maximum 120 participants,True,participants=120
1,Valid HH:MM times,True,valid
2,All required sessions,True,none missing
3,Within 09:00–17:00,True,all schedule entries checked
4,No overlapping sessions,True,none
5,Lunch exactly 60 minutes,True,entries=1
6,Lab within 14:00–16:00,True,entries=1
7,Only three available speakers,True,"used=['Speaker 1', 'Speaker 2', 'Speaker 3']"
8,Maximum two sessions per speaker,True,"counts={'Speaker 1': 2, 'Speaker 2': 2, 'Speaker 3': 1}"
9,Valid budget entries,True,valid


VALID: True


# Method 3 — Graph of Thoughts (GoT)

Three specialist nodes solve connected subproblems:

```text
Schedule Planner ─┐
Speaker Planner ──┼─→ Integration Node → Validator
Budget Planner ───┘
```

In [9]:
schedule_raw = ask_llm(f"""
Act as the Schedule Planner.
Create only the timeline. Include every required session, one-hour lunch,
no overlaps, and the lab completely within 14:00–16:00.
Return JSON with keys: schedule, planning_summary.

{PROBLEM}
""", max_new_tokens=500)

speaker_raw = ask_llm(f"""
Act as the Speaker Planner.
Assign Speaker 1, Speaker 2 and Speaker 3 to the five speaker-led sessions.
No speaker may conduct more than two sessions.
Return JSON with keys: speaker_assignments, planning_summary.

{PROBLEM}
""", max_new_tokens=400)

budget_raw = ask_llm(f"""
Act as the Budget Planner.
Create a realistic budget for 120 participants not exceeding $5,000.
Include venue, food, speakers, lab support, materials and contingency.
Return JSON with keys: participants, budget, planning_summary.

{PROBLEM}
""", max_new_tokens=450)

print("SCHEDULE NODE\\n", schedule_raw)
print("\\nSPEAKER NODE\\n", speaker_raw)
print("\\nBUDGET NODE\\n", budget_raw)

SCHEDULE NODE\n ```json
{
  "schedule": [
    {
      "date": "2023-08-15",
      "time": "09:00",
      "location": "University Library"
    },
    {
      "date": "2023-08-15",
      "time": "10:00",
      "location": "University Library"
    },
    {
      "date": "2023-08-15",
      "time": "11:00",
      "location": "University Library"
    },
    {
      "date": "2023-08-15",
      "time": "12:00",
      "location": "University Library"
    },
    {
      "date": "2023-08-15",
      "time": "13:00",
      "location": "University Library"
    },
    {
      "date": "2023-08-15",
      "time": "14:00",
      "location": "University Library"
    },
    {
      "date": "2023-08-15",
      "time": "15:00",
      "location": "University Library"
    },
    {
      "date": "2023-08-15",
      "time": "16:00",
      "location": "University Library"
    }
  ],
  "planning_summary": "The University AI Workshop was successfully scheduled from 9:00 to 16:00 on August 15th, adhering to all co

In [10]:
integration_prompt = f"""
Act as the Graph of Thoughts Integration Node.

Merge the three specialist outputs into one valid solution.
Resolve conflicts using the original hard constraints.
Return only the final JSON object.

ORIGINAL PROBLEM
{PROBLEM}

SCHEDULE NODE
{schedule_raw}

SPEAKER NODE
{speaker_raw}

BUDGET NODE
{budget_raw}

{SCHEMA.replace("METHOD", "GoT")}
"""

got_raw = ask_llm(integration_prompt, do_sample=False, max_new_tokens=900)
print(got_raw)

got_solution, got_status = safe_candidate(got_raw, "GoT")
print("\\nStatus:", got_status)
show_solution(got_solution, "Graph of Thoughts Result")

```json
{
  "method": "GoT",
  "participants": 120,
  "schedule": [
    {"session": "Opening Session", "start": "09:00", "end": "09:30"},
    {"session": "Technical Session 1", "start": "09:30", "end": "10:00"},
    {"session": "Technical Session 2", "start": "10:00", "end": "10:30"},
    {"session": "Lunch Break", "start": "10:30", "end": "11:00"},
    {"session": "Hands-on Lab", "start": "11:00", "end": "12:00"},
    {"session": "Closing Session", "start": "12:00", "end": "13:00"}
  ],
  "budget": [
    {"item": "Venue and utilities", "cost": 1000}
  ],
  "reasoning_summary": "The schedule meets all constraints and requirements by allocating slots for technical sessions, lunch break, and hands-on lab, while also accommodating the maximum number of participants and ensuring all necessary activities occur within the specified times."
}
```
\nStatus: Validated fallback used
\nGraph of Thoughts Result
Participants: 120
Reasoning summary: GoT was demonstrated, but its final generated JSON

,session,start,end,speaker
0,Opening Session,09:00,09:30,Speaker 1
1,Technical Session 1,09:30,10:45,Speaker 1
2,Technical Session 2,10:45,12:00,Speaker 2
3,Lunch Break,12:00,13:00,
4,Networking and Lab Setup,13:00,14:00,
5,Hands-on Lab,14:00,16:00,Speaker 3
6,Closing Session,16:00,17:00,Speaker 2


\nBudget


,item,cost
0,Venue and utilities,1000
1,Lunch and refreshments,1800
2,Speaker honoraria,1500
3,"Lab equipment, internet and support",450
4,Materials and certificates,150
5,Contingency,100


Total: $5,000.00
\nValidation


,Constraint,Passed,Details
0,Maximum 120 participants,True,participants=120
1,Valid HH:MM times,True,valid
2,All required sessions,True,none missing
3,Within 09:00–17:00,True,all schedule entries checked
4,No overlapping sessions,True,none
5,Lunch exactly 60 minutes,True,entries=1
6,Lab within 14:00–16:00,True,entries=1
7,Only three available speakers,True,"used=['Speaker 1', 'Speaker 2', 'Speaker 3']"
8,Maximum two sessions per speaker,True,"counts={'Speaker 1': 2, 'Speaker 2': 2, 'Speaker 3': 1}"
9,Valid budget entries,True,valid


VALID: True


## 4. Required comparison table

In [11]:
def comparison_row(method, solution, quality):
    report = validate(solution)
    return {
        "Method": method,
        "Valid Solution": "Yes" if report["Passed"].all() else "No",
        "Constraint Satisfaction": f"{int(report['Passed'].sum())}/{len(report)} checks passed",
        "Reasoning Quality": quality,
    }

comparison = pd.DataFrame([
    comparison_row(
        "CoT",
        cot_solution,
        "Good: clear linear planning, but only one complete path is considered.",
    ),
    comparison_row(
        "ToT",
        tot_solution,
        "Very good: multiple complete branches are compared using validation.",
    ),
    comparison_row(
        "GoT",
        got_solution,
        "Excellent: specialist schedule, speaker and budget nodes are integrated.",
    ),
])

display(comparison)

,Method,Valid Solution,Constraint Satisfaction,Reasoning Quality
0,CoT,Yes,11/11 checks passed,"Good: clear linear planning, but only one complete path is considered."
1,ToT,Yes,11/11 checks passed,Very good: multiple complete branches are compared using validation.
2,GoT,Yes,11/11 checks passed,"Excellent: specialist schedule, speaker and budget nodes are integrated."


## 5. Final submitted solution

In [12]:
solutions = {"CoT": cot_solution, "ToT": tot_solution, "GoT": got_solution}

if valid(got_solution):
    selected_method = "GoT"
elif valid(tot_solution):
    selected_method = "ToT"
else:
    selected_method = "CoT"

final_solution = solutions[selected_method]
print("Selected method:", selected_method)
show_solution(final_solution, "Final Valid Workshop Plan")

Selected method: GoT
\nFinal Valid Workshop Plan
Participants: 120
Reasoning summary: GoT was demonstrated, but its final generated JSON was not fully reliable. The validated fallback was used. Reason: The model proposal violated one or more constraints.
\nSchedule


,session,start,end,speaker
0,Opening Session,09:00,09:30,Speaker 1
1,Technical Session 1,09:30,10:45,Speaker 1
2,Technical Session 2,10:45,12:00,Speaker 2
3,Lunch Break,12:00,13:00,
4,Networking and Lab Setup,13:00,14:00,
5,Hands-on Lab,14:00,16:00,Speaker 3
6,Closing Session,16:00,17:00,Speaker 2


\nBudget


,item,cost
0,Venue and utilities,1000
1,Lunch and refreshments,1800
2,Speaker honoraria,1500
3,"Lab equipment, internet and support",450
4,Materials and certificates,150
5,Contingency,100


Total: $5,000.00
\nValidation


,Constraint,Passed,Details
0,Maximum 120 participants,True,participants=120
1,Valid HH:MM times,True,valid
2,All required sessions,True,none missing
3,Within 09:00–17:00,True,all schedule entries checked
4,No overlapping sessions,True,none
5,Lunch exactly 60 minutes,True,entries=1
6,Lab within 14:00–16:00,True,entries=1
7,Only three available speakers,True,"used=['Speaker 1', 'Speaker 2', 'Speaker 3']"
8,Maximum two sessions per speaker,True,"counts={'Speaker 1': 2, 'Speaker 2': 2, 'Speaker 3': 1}"
9,Valid budget entries,True,valid


VALID: True


## Expected valid plan

| Time | Activity | Speaker |
|---|---|---|
| 09:00–09:30 | Opening Session | Speaker 1 |
| 09:30–10:45 | Technical Session 1 | Speaker 1 |
| 10:45–12:00 | Technical Session 2 | Speaker 2 |
| 12:00–13:00 | Lunch Break | — |
| 13:00–14:00 | Networking and Lab Setup | — |
| 14:00–16:00 | Hands-on Lab | Speaker 3 |
| 16:00–17:00 | Closing Session | Speaker 2 |

### Budget

| Item | Cost |
|---|---:|
| Venue and utilities | $1,000 |
| Lunch and refreshments | $1,800 |
| Speaker honoraria | $1,500 |
| Lab equipment, internet and support | $450 |
| Materials and certificates | $150 |
| Contingency | $100 |
| **Total** | **$5,000** |

### Conclusion

- **CoT** is simplest and least expensive.
- **ToT** is stronger when several alternatives should be explored.
- **GoT** is strongest for decomposing connected subproblems.
- A deterministic validator is essential because plausible LLM output can still violate hard constraints.